# RGBD 4-Channel Fusion — Single Sample Alignment Check

Pushes one triplet (`.bin`, `_texture.png`, `.png`) through preprocessing and builds the
**aligned 4-channel (RGB + Depth) input** that the `rgbd_early` early-fusion Mask R-CNN
needs: depth comes from the contour render and is already aligned via the RGB crop path.

The final cell overlays the depth channel on the RGB to verify co-registration with the
RGB image and the binary mask.

In [ ]:
# ── SET THESE THREE PATHS ─────────────────────────────────────────────────
BIN_PATH       = ""   # <sample>.bin           (point cloud / depth)
TEXTURE_PATH   = ""   # <sample>_texture.png   (raw RGB image)
SEGMENTED_PATH = ""   # <sample>.png           (red scanner segmentation → mask)
# ─────────────────────────────────────────────────────────────────────────

In [ ]:
import os, sys, importlib
import numpy as np
import cv2
import matplotlib.pyplot as plt

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
UTIL_DIR = NOTEBOOK_DIR
if UTIL_DIR not in sys.path:
    sys.path.insert(0, UTIL_DIR)

import preprocessing.TumorDataset.tumor_dataset as tumor_dataset
importlib.reload(tumor_dataset)
from preprocessing.TumorDataset.tumor_dataset import Dataset
from preprocessing.preprocessing_functions._io import read_bin

In [ ]:
# ── Read the triplet ──────────────────────────────────────────────────────
rgb_img = cv2.imread(TEXTURE_PATH)        # _texture.png  -> RGB base image
seg_img = cv2.imread(SEGMENTED_PATH)      # .png          -> red scanner segmentation (mask source)

# Read the point cloud from the binary file
grid_x, grid_y, grid_z = read_bin(BIN_PATH)
depth_info = [(grid_x, grid_y, grid_z, os.path.basename(BIN_PATH))]

# The Dataset mixin methods operate on lists; wrap the single sample
rgb = [rgb_img]
seg = [seg_img]

# Instantiate Dataset as a method host
ds = Dataset(data_path=os.path.dirname(TEXTURE_PATH))

In [ ]:
# ── Build RGB, mask, the aligned depth channel, and the 4-channel stack ────
og_masks = [m.copy() for m in seg]

# --- RGB (crop_raw_images → pad → crop; mask: zoom → offset → binarize) ---
images_rgb = ds.crop_raw_images(rgb)
masks      = ds.crop_masks(seg)
images_rgb, masks = ds.add_padding(images_rgb, masks)
masks      = ds.zoom_at(masks, 1.333, coord=None)
images_rgb = ds.crop_images(images_rgb)
masks      = ds.crop_images_offset(masks, x_offset=-25)
masks      = ds.create_binary_masks(masks)
masks      = ds.correct_binary_masks(masks)

# --- Depth: CONTOUR (aligned — same crop path as RGB) ---
masks_clone_c = ds.crop_masks([m.copy() for m in og_masks])
depth_contour = ds.read_contours_array_depth(depth_info)
depth_contour = ds.crop_raw_images(depth_contour)
depth_contour, masks_clone_c = ds.add_padding(depth_contour, masks_clone_c)
depth_contour = ds.crop_images(depth_contour)

# --- Single depth channel (shared convention) + 4-channel stack ---
dch_early  = ds._contour_to_depth_channel(depth_contour[0])   # (256,256) f32
rgbd_early = np.dstack([images_rgb[0], dch_early])            # (256,256,4)

print("Done.")
for n, a in [("rgb", images_rgb[0]), ("mask", masks[0]),
             ("early 4ch", rgbd_early)]:
    print(f"  {n:<12}: {a.shape}")

In [ ]:
# ── Alignment verification for the rgbd_early 4-channel input ──────────────
# Columns: RGB | depth channel | depth-on-RGB overlay | binary mask
# A valid 4-ch input => the depth's dark tumor blob sits on the RGB tumor and
# inside the white mask. Mis-registration is obvious in the overlay column.

rgb_disp = cv2.cvtColor(images_rgb[0], cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle(os.path.basename(TEXTURE_PATH), fontsize=12)

axes[0].imshow(rgb_disp)
axes[0].set_title("rgbd_early\nRGB")

axes[1].imshow(dch_early, cmap="gray")
axes[1].set_title("Depth channel")

axes[2].imshow(rgb_disp)
axes[2].imshow(dch_early, cmap="inferno", alpha=0.5)
axes[2].set_title("Depth ⊕ RGB overlay")

axes[3].imshow(masks[0], cmap="gray")
axes[3].set_title("Binary mask")

for c in range(4):
    axes[c].axis("off")

plt.tight_layout()
plt.show()